In [5]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import os
import glob
import yaml

DATASET_DIR = "/kaggle/input/homeobject3k"
TRAIN_IMAGES_DIR = f"{DATASET_DIR}/train/images"
TRAIN_LABELS_DIR = f"{DATASET_DIR}/train/labels"
VALID_IMAGES_DIR = f"{DATASET_DIR}/valid/images"
VALID_LABELS_DIR = f"{DATASET_DIR}/valid/labels"

SPLIT_YAML = "/kaggle/working/homeobject3k_split.yaml"

model = YOLO("yolo11n.pt")

In [ ]:
# Use existing train/valid split and create YAML
train_images = sorted(glob.glob(f"{TRAIN_IMAGES_DIR}/*"))
valid_images = sorted(glob.glob(f"{VALID_IMAGES_DIR}/*"))

if len(train_images) == 0:
    raise RuntimeError(f"No training images found in {TRAIN_IMAGES_DIR}")
if len(valid_images) == 0:
    raise RuntimeError(f"No validation images found in {VALID_IMAGES_DIR}")

class_ids = set()
for lbl_path in glob.glob(f"{TRAIN_LABELS_DIR}/*.txt") + glob.glob(f"{VALID_LABELS_DIR}/*.txt"):
    with open(lbl_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                class_ids.add(int(line.split()[0]))

max_id = max(class_ids) if len(class_ids) > 0 else -1
names = [f"class_{i}" for i in range(max_id + 1)]

split_cfg = {
    "train": TRAIN_IMAGES_DIR,
    "val": VALID_IMAGES_DIR,
    "names": names
}

with open(SPLIT_YAML, "w") as f:
    yaml.safe_dump(split_cfg, f, sort_keys=False)

print("YAML ready:", SPLIT_YAML)
print("Train images:", len(train_images))
print("Val images:", len(valid_images))
print("Classes:", len(names))

In [ ]:
# Train with simple augmentation settings
results = model.train(
    data=SPLIT_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    degrees=12.0,
    translate=0.10,
    scale=0.40,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.70,
    hsv_v=0.40
)

Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=HomeObjects-3K.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspectiv

Exception in thread Thread-19 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

KeyboardInterrupt: 

In [ ]:
val_model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

metrics = val_model.val(data=SPLIT_YAML)

print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_dir = '/kaggle/working/runs/detect/train/'

plt.figure(figsize=(10, 10))
img = mpimg.imread(os.path.join(results_dir, 'confusion_matrix.png'))
plt.imshow(img)
plt.axis('off')
plt.title('Confusion Matrix')
plt.show()

plt.figure(figsize=(12, 8))
img = mpimg.imread(os.path.join(results_dir, 'results.png'))
plt.imshow(img)
plt.axis('off')
plt.title('Training Metrics and Loss Curves')
plt.show()